# Vuln2Action - hybrid two-stage evaluation

End-to-end evaluation for:

```text
CVE description -> predicted CAPEC -> CAPEC-conditioned T5 -> execution flow
```

Evaluation views:

- `common_predicted`: the fixed 1,000-row baseline validation set. The labels were produced by the same Word2Vec selector, so this is an in-distribution check only.
- `reference_gold`: independent CVE2CAPEC reference CAPEC supplied as input; upper bound for stage 2.
- `reference_predicted`: predicted CAPEC supplied as input and evaluated against a valid reference flow; end-to-end.
- `reference_predicted_correct`: only rows where the predicted CAPEC is in the independent reference set; stage-2 conditional score.
- `reference_cve_only`: CVE-only model on the same independent-reference targets; lower-bound comparison.

No gold CAPEC is put into the hybrid input. Only the existing selector output is used.


In [ ]:
!nvidia-smi
import torch
print("torch:", torch.__version__)
print("cuda devices:", torch.cuda.device_count())


In [ ]:
!rm -rf /kaggle/working/Vuln2Action-Lab
!git clone --branch feature/paper-reproduction https://github.com/Hoang-Minh-Hieu-2806/Vuln2Action-Lab.git /kaggle/working/Vuln2Action-Lab
%cd /kaggle/working/Vuln2Action-Lab
!pip install -q -e ".[train]"


In [ ]:
from pathlib import Path
import glob, os, sys, json, subprocess

BUNDLE = Path("/kaggle/input/vuln2action-hybrid-eval-bundle")
print("bundle exists:", BUNDLE.exists())
for p in sorted(BUNDLE.glob("*")):
    print("bundle entry:", p)

def find_model(inner_name):
    hits = sorted(glob.glob(f"/kaggle/input/**/{inner_name}/final/config.json", recursive=True))
    print(f"{inner_name} hits:", hits)
    if not hits:
        raise FileNotFoundError(f"model not found: {inner_name}")
    return Path(hits[0]).parent

CAPEC_CKPT = find_model("t5_stage1_capec_conditioned")
CVE_CKPT = find_model("t5_stage1_target_steps_pilot")
print("CAPEC_CKPT:", CAPEC_CKPT)
print("CVE_CKPT:", CVE_CKPT)


In [ ]:
OUT = Path("/kaggle/working/hybrid_eval")
OUT.mkdir(parents=True, exist_ok=True)

def run_eval(label, dataset, checkpoint, include_capec):
    dataset = Path(dataset)
    assert (dataset / "flow_validation.jsonl").is_file(), dataset
    output = OUT / label
    output.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable, "scripts/evaluate_t5.py",
        "--checkpoint", str(checkpoint),
        "--dataset", str(dataset),
        "--split", "validation",
        "--target-field", "target_steps",
        "--output-dir", str(output),
        "--max-input-length", "256",
        "--max-target-length", "384",
        "--batch-size", "4",
        "--num-beams", "4",
        "--fp16",
    ]
    cmd.append("--include-capec" if include_capec else "--no-include-capec")
    print("\nRUN", label, "include_capec=", include_capec)
    print(" ".join(cmd))
    subprocess.run(cmd, check=True, cwd="/kaggle/working/Vuln2Action-Lab")
    metrics = json.loads((output / "metrics.json").read_text(encoding="utf-8"))
    print(label, "BLEU=", metrics["bleu"], "ROUGE-L F1=", metrics["rougeL_f1"])

common = BUNDLE / "hybrid_eval" / "common_predicted"
random = BUNDLE / "hybrid_eval_random1000"

run_eval("common_predicted", common, CAPEC_CKPT, True)
run_eval("random_reference_gold", random / "reference_gold", CAPEC_CKPT, True)
run_eval("random_reference_predicted", random / "reference_predicted", CAPEC_CKPT, True)
run_eval("random_reference_predicted_correct", random / "reference_predicted_correct", CAPEC_CKPT, True)
run_eval("random_reference_cve_only", random / "reference_cve_only", CVE_CKPT, False)


In [ ]:
import json
from pathlib import Path
summary = {}
for p in sorted(Path("/kaggle/working/hybrid_eval").glob("*/metrics.json")):
    m = json.loads(p.read_text(encoding="utf-8"))
    summary[p.parent.name] = {
        "rows": m["rows"],
        "bleu": m["bleu"],
        "bleu_corpus": m["bleu_corpus"],
        "rouge1_f1": m["rouge1_f1"],
        "rouge2_f1": m["rouge2_f1"],
        "rougeL_f1": m["rougeL_f1"],
        "include_capec": m["include_capec"],
    }
print(json.dumps(summary, indent=2, ensure_ascii=False))
Path("/kaggle/working/hybrid_eval/summary.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8"
)
for p in sorted(Path("/kaggle/working/hybrid_eval").glob("*/predictions.jsonl")):
    print("predictions:", p, p.stat().st_size)
